[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/07_alumno_etica.ipynb)

# MLY1101 · Machine Learning — Actividad 1.4
## Impacto ético, sesgos y privacidad

**Resultado de aprendizaje (RA1):** recopila, a través de un trabajo colaborativo, sets de datos
representativos y de calidad, a partir de distintas fuentes, para responder a las necesidades del
contexto de negocio, **considerando aspectos éticos**.

**Indicador de logro (IL 1.4):** evalúa el impacto ético y los sesgos en los datos recopilados,
garantizando el cumplimiento de estándares de privacidad en el manejo de información.

---

### Por qué esta actividad no es el apéndice del curso

La ética de los datos se suele dejar para la última diapositiva, en forma de advertencias
generales que nadie puede aplicar. Hoy vamos a hacer lo contrario: **medir**.

Un sesgo sin una cifra que lo respalde es una opinión. Un riesgo de privacidad sin un caso
concreto es un trámite. En esta sesión, cada afirmación ética va a tener un número al lado.

> **La idea central:** un algoritmo no es neutral. Replica y **amplifica** las asimetrías que
> había en los datos con los que se entrenó, y lo hace en silencio, porque la métrica promedio
> se ve bien.

---

### Las tres cosas que vamos a medir

| Bloque | Pregunta | Cómo se responde |
|---|---|---|
| **Sesgo de muestreo** | ¿Los datos representan al mundo donde va a operar el sistema? | Censo completo de las condiciones de grabación |
| **Sesgo de procesamiento** | ¿Mis decisiones de limpieza perjudican a un grupo? | Composición del dataset antes y después |
| **Privacidad** | ¿Se puede reidentificar a alguien con estos datos? | Cardinalidad de combinaciones de columnas |

---

### Al final de la sesión debes entregar

Una **ficha del dataset** (*datasheet*) con la evaluación de impacto: quién está
subrepresentado y con qué cifra, qué decisión técnica lo empeoraría, sobre quién recae la
consecuencia, y qué datos personales hay o podrían reconstruirse.

---
## Preparación del entorno

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"

print("Colab:", EN_COLAB, "| dataset:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import eda

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

df = pd.read_csv(RUTA_DATOS)
print(f"{df.shape[0]:,} detecciones en {df['segment_id'].nunique()} segmentos")

---
# Bloque 1 · ⭐⭐ Sesgo de muestreo: ¿a quién no vio el sensor?

**Sesgo de muestreo** es que los datos no representen proporcionalmente al mundo donde el
sistema va a operar. No es un error de cálculo: es una consecuencia de **dónde y cuándo** se
recolectó.

No vamos a discutirlo en abstracto. El Waymo Open Dataset publica las condiciones de grabación
de todos sus segmentos, y en este repositorio está el **censo completo** —los 798 segmentos del
conjunto de entrenamiento, no una muestra— en `docs/sesgo_waymo.md`.

### ✏️ TODO 1 — Antes de mirar, apuesta

El Waymo Open Dataset tiene **798 segmentos** de entrenamiento, grabados por una flota de
vehículos autónomos en Estados Unidos.

**¿Cuántos crees que se grabaron con lluvia?** `____`

*(Escríbelo antes de seguir. No hagas trampa.)*

In [ ]:
# El censo real, medido sobre los 798 segmentos (ver docs/sesgo_waymo.md).
censo = pd.DataFrame(
    [
        {"condicion": "clima", "valor": "soleado", "segmentos": 793},
        {"condicion": "clima", "valor": "lluvia", "segmentos": 5},
        {"condicion": "momento", "valor": "día", "segmentos": 647},
        {"condicion": "momento", "valor": "noche", "segmentos": 79},
        {"condicion": "momento", "valor": "amanecer/atardecer", "segmentos": 72},
        {"condicion": "lugar", "valor": "San Francisco", "segmentos": 409},
        {"condicion": "lugar", "valor": "Phoenix", "segmentos": 284},
        {"condicion": "lugar", "valor": "otras", "segmentos": 105},
    ]
)
censo["pct"] = censo.groupby("condicion")["segmentos"].transform(lambda s: 100 * s / s.sum()).round(1)
censo

### ✏️ TODO 2 — Ponerlo en palabras que signifiquen algo

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. Completa la frase con la cifra: *"Un vehículo autónomo entrenado con estos datos ha visto
   llover ____ veces."*
2. ¿Qué pasa cuando ese sistema se despliega en Santiago en junio?
3. **La pregunta difícil:** ¿es esto un error de Waymo?

---
# Bloque 2 · ¿Quién es minoría dentro del dataset?

El sesgo de muestreo es sobre las **condiciones**. Ahora miremos los **sujetos**: qué objetos
aparecen y en qué proporción.

### ✏️ TODO 3 — La composición

Calcula la frecuencia de cada tipo de objeto, **después de unificar las variantes de escritura**
(si no, los peatones quedan repartidos en cuatro grafías y ninguna cifra sirve).

In [ ]:
# TODO 3: composición por tipo de objeto, tras unificar variantes.
tipos = eda.____(
    df["object_type"], mapa={"peaton": "pedestrian", "ped": "pedestrian"}
)
composicion = eda.____(tipos)
composicion

### ✏️ TODO 4 — Del porcentaje a la persona

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. ¿Qué tipo de objeto es la minoría, y con qué porcentaje?
2. En la vía pública, ¿quién es más vulnerable ante un error de detección: el ocupante de un
   vehículo o esa minoría? ¿Por qué?
3. Une las dos respuestas en una frase.

In [ ]:
# Autochequeo
minoritaria = composicion["pct"].idxmin()
pct = composicion.loc[minoritaria, "pct"]
assert pct < 5, "revisa: ¿normalizaste las categorías antes de contar?"
print(f"✅ Clase minoritaria: {minoritaria} con {pct:.2f} % de las detecciones.")
print(f"   Razón respecto de la mayoritaria: {composicion.loc[minoritaria, 'ratio_vs_mayoritaria']:.4f}")
print("   Es decir, por cada ciclista hay unos", int(1/composicion.loc[minoritaria, 'ratio_vs_mayoritaria']), "vehículos.")

---
# Bloque 3 · ⭐⭐ Tu limpieza también sesga

Los dos bloques anteriores son sobre datos que llegaron así. Este es sobre **lo que tú les
haces**.

`dropna()` es la operación más inocente del análisis de datos. Elimina filas incompletas. Nadie
la discute.

Vamos a ver a quién elimina.

### ✏️ TODO 5 — ¿Cuánto se pierde en total?

Empieza por la cifra que cualquiera reportaría.

In [ ]:
# TODO 5: ¿cuántas filas se perderían con un dropna sobre speed_mps?
perdidas = df["speed_mps"].____().sum()
print(f"Filas con speed_mps faltante: {perdidas:,} de {len(df):,} "
      f"({100*perdidas/len(df):.2f} % del dataset)")

Menos del 2 %. En cualquier informe eso se describiría como *"se eliminaron unas pocas filas
incompletas"* y nadie preguntaría más.

### ✏️ TODO 6 — La misma cifra, por grupo

Ahora calcula el porcentaje de faltantes **según el momento del día**.

In [ ]:
# TODO 6: ¿el faltante se reparte igual entre los grupos?
por_momento = eda.____(df, "speed_mps", ["____"])
print(por_momento.sort_values("pct_nulos", ascending=False), "\n")

peor = por_momento["pct_nulos"].idxmax()
mejor = por_momento["pct_nulos"].idxmin()
factor = por_momento.loc[peor, "pct_nulos"] / por_momento.loc[mejor, "pct_nulos"]
print(f"'{peor}' pierde {factor:.1f} veces más filas que '{mejor}'.")

### ✏️ TODO 7 — El efecto sobre la composición

Compara cómo se reparte el dataset por momento del día **antes y después** del `dropna()`.

In [ ]:
# TODO 7: ¿cómo cambia la composición del dataset tras el dropna?
antes = df["time_of_day"].value_counts(normalize=True).mul(100)
despues = df.____(subset=["speed_mps"])["time_of_day"].value_counts(normalize=True).mul(100)

efecto = pd.DataFrame({"antes_%": antes.round(2), "tras_dropna_%": despues.round(2)})
efecto["cambio_pp"] = (efecto["tras_dropna_%"] - efecto["antes_%"]).round(2)
efecto["pct_del_grupo_perdido"] = (
    100 * (df["time_of_day"].value_counts() - df.dropna(subset=["speed_mps"])["time_of_day"].value_counts())
    / df["time_of_day"].value_counts()
).round(2)
efecto

In [ ]:
# Autochequeo
assert factor > 2, "revisa: la diferencia entre grupos debería ser grande, no marginal"
assert efecto.loc["Night", "cambio_pp"] < 0, "la noche debería PERDER peso tras el dropna"
print(f"✅ El dropna borra el {100*perdidas/len(df):.2f} % del dataset...")
print(f"   ...pero el {efecto.loc['Night', 'pct_del_grupo_perdido']:.2f} % de las detecciones nocturnas,")
print(f"   contra el {efecto.loc['Dawn/Dusk', 'pct_del_grupo_perdido']:.2f} % de las del amanecer.")
print()
print("   Una operación que en el informe aparece como 'se limpiaron los datos'")
print("   acaba de sesgar el dataset contra la condición peor medida.")

### ✏️ TODO 8 — El eslabón que casi nadie ve

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

Ordena esta cadena y complétala:

```
De noche el LiDAR recibe menos puntos
        ↓
la velocidad se estima peor y se registra como faltante más seguido
        ↓
un dropna() elimina proporcionalmente más detecciones nocturnas
        ↓
        ____
        ↓
        ____
        ↓
y la métrica de evaluación NO lo muestra. ¿Por qué?
```

La última pregunta es la importante.

### ✏️ TODO 9 — La alternativa

Alguien propone: *"entonces imputamos la velocidad con la media y listo"*.

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

¿Resuelve el problema? ¿Qué le harías tú en cambio?

---
# Bloque 4 · ⭐ Privacidad: tres columnas inocentes

Este dataset no tiene nombres, ni RUT, ni correos. ¿Está entonces libre de riesgo de privacidad?

La respuesta correcta no es sí ni no: es **depende de qué se puede reconstruir combinando
columnas**. A eso se le llama **reidentificación**, y es el motivo por el que "anonimizar"
borrando la columna del nombre casi nunca basta.

### Marco legal en Chile

La **Ley 19.628** sobre protección de la vida privada, reformada por la **Ley 21.719** (2024),
que crea la Agencia de Protección de Datos Personales. Dos principios que aplican directamente
a lo que hacemos hoy:

- **Minimización:** recolectar solo lo necesario para la finalidad declarada. *Lo que no se
  recolecta no se filtra.*
- **Finalidad:** los datos se usan para aquello que se declaró al obtenerlos, no para lo que se
  nos ocurra después.

### ✏️ TODO 10 — ¿Cuántas columnas hacen falta para señalar a uno solo?

Calcula, para varias combinaciones de columnas, qué porcentaje de los grupos resultantes
contiene **una sola detección**. Un grupo de tamaño 1 es una fila señalada de forma única.

In [ ]:
# TODO 10: ¿cuántas columnas hacen falta para aislar una sola detección?
combinaciones = [
    ["segment_id"],
    ["segment_id", "time_of_day"],
    ["segment_id", "timestamp_micros"],
    ["segment_id", "timestamp_micros", "object_type"],
]

filas = []
for columnas in combinaciones:
    tamanos = df.____(columnas).size()
    filas.append(
        {
            "combinacion": " + ".join(columnas),
            "n_columnas": len(columnas),
            "grupos": len(tamanos),
            "grupos_de_una_fila": int((tamanos == ____).sum()),
            "pct_unicos": round(100 * (tamanos == 1).mean(), 1),
        }
    )
pd.DataFrame(filas)

In [ ]:
# Autochequeo
tres = df.groupby(["segment_id", "timestamp_micros", "object_type"]).size()
assert (tres == 1).mean() > 0.5, "revisa la combinación de tres columnas"
print(f"✅ Con tres columnas, ninguna identificadora por sí sola,")
print(f"   el {100*(tres==1).mean():.1f} % de las combinaciones señala UNA sola detección.")
print()
print("   Ninguna de las tres es un dato personal. Las tres juntas son un identificador.")

### ✏️ TODO 11 — La lista de chequeo, aplicada

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. Este dataset describe objetos, no personas. Pero **un peatón detectado es una persona**.
   ¿Qué se podría reconstruir sobre ella si se cruzara con otra fuente?
2. Aplicando **minimización**: ¿qué columna quitarías si el objetivo es solo clasificar el tipo
   de objeto?
3. El notebook opcional trabaja con datos **reales** de Waymo, cuya licencia prohíbe
   redistribuirlos. ¿Por qué crees que existe esa restricción?

---
# Bloque 5 · La ficha del dataset

Documentar no es el trámite de después: es lo que hace utilizable un dataset sesgado. Un
*datasheet* responde las preguntas que alguien necesitará dentro de dos años, cuando quien lo
armó ya no esté.

Esta es la entrega de la Actividad 1.4. Rellénala **para el caso oficial que eligió tu equipo**
(Telco, Housing o Spotify); el ejemplo de arriba es la referencia de cómo se hace.

---

## Ficha del dataset

**Dataset:** `____` · **Equipo:** `____` · **Fecha:** `____`

### Origen

| Campo | Valor |
|---|---|
| Quién lo recolectó | `____` |
| Con qué propósito original | `____` |
| Cómo se recolectó (muestreo, censo, scraping…) | `____` |
| Periodo y lugar | `____` |
| Licencia y usos permitidos | `____` |

### Representatividad

**Población que el sistema debería cubrir:** `____`
**Población que el dataset realmente cubre:** `____`

| Grupo | % en el dataset | % esperado en el mundo | Brecha |
|---|---|---|---|
| `____` | | | |
| `____` | | | |

**Grupo subrepresentado, con cifra:** `____`

> No vale "podría haber sesgo". Vale *"el grupo X es el 4 % de las filas y concentra el 18 % de
> los valores faltantes"*.

### Evaluación de impacto

**Decisión técnica que empeoraría el sesgo:** `____`
**Consecuencia concreta, y sobre quién recae:** `____`
**Por qué la métrica promedio no lo mostraría:** `____`
**Qué haremos al respecto:** `____`

### Privacidad

- [ ] Identificadores directos: `____`
- [ ] **Combinación de columnas que permite reidentificar:** `____` *(con su % de únicos)*
- [ ] Columnas que quitamos por minimización: `____`
- [ ] Licencia verificada, no supuesta: `____`
- [ ] Podemos declarar origen y fecha: `____`

### Usos para los que este dataset **no** sirve

`____`

*(La sección más útil de la ficha y la que nunca se escribe. Un dataset acotado y declarado es
profesional; uno presentado como universal, no.)*